In [0]:
CREATE OR REPLACE TABLE nyc_mobility.validation.weather_validation AS

WITH total_rows AS (
    SELECT COUNT(*) AS total_rows
    FROM nyc_mobility.clean.weather_silver
),

dq_results AS (

-- LATITUDE

SELECT
    'latitude' AS column_name,
    'Completeness' AS data_quality_check,
    COUNT(*) AS failed_rows
FROM nyc_mobility.clean.weather_silver
WHERE latitude IS NULL

UNION ALL

SELECT
    'latitude',
    'Validity',
    COUNT(*)
FROM nyc_mobility.clean.weather_silver
WHERE latitude NOT BETWEEN -90 AND 90

-- LONGITUDE

UNION ALL

SELECT
    'longitude',
    'Completeness',
    COUNT(*)
FROM nyc_mobility.clean.weather_silver
WHERE longitude IS NULL

UNION ALL

SELECT
    'longitude',
    'Validity',
    COUNT(*)
FROM nyc_mobility.clean.weather_silver
WHERE longitude NOT BETWEEN -180 AND 180

-- OBSERVATION TIME

UNION ALL

SELECT
    'observation_time',
    'Completeness',
    COUNT(*)
FROM nyc_mobility.clean.weather_silver
WHERE observation_time IS NULL

UNION ALL

SELECT
    'observation_time',
    'Timeliness',
    COUNT(*)
FROM nyc_mobility.clean.weather_silver
WHERE observation_time > CURRENT_TIMESTAMP()

-- PRECIPITATION

UNION ALL

SELECT
    'precipitation',
    'Completeness',
    COUNT(*)
FROM nyc_mobility.clean.weather_silver
WHERE precipitation IS NULL

UNION ALL

SELECT
    'precipitation',
    'Accuracy',
    COUNT(*)
FROM nyc_mobility.clean.weather_silver
WHERE precipitation < 0

-- RAIN

UNION ALL

SELECT
    'rain',
    'Completeness',
    COUNT(*)
FROM nyc_mobility.clean.weather_silver
WHERE rain IS NULL

UNION ALL

SELECT
    'rain',
    'Accuracy',
    COUNT(*)
FROM nyc_mobility.clean.weather_silver
WHERE rain < 0

-- SNOWFALL

UNION ALL

SELECT
    'snowfall',
    'Completeness',
    COUNT(*)
FROM nyc_mobility.clean.weather_silver
WHERE snowfall IS NULL

UNION ALL

SELECT
    'snowfall',
    'Accuracy',
    COUNT(*)
FROM nyc_mobility.clean.weather_silver
WHERE snowfall < 0

-- TEMPERATURE

UNION ALL

SELECT
    'temperature_2m',
    'Completeness',
    COUNT(*)
FROM nyc_mobility.clean.weather_silver
WHERE temperature_2m IS NULL

UNION ALL

SELECT
    'temperature_2m',
    'Validity',
    COUNT(*)
FROM nyc_mobility.clean.weather_silver
WHERE temperature_2m < -90
   OR temperature_2m > 60

-- TIMEZONE

UNION ALL

SELECT
    'timezone',
    'Completeness',
    COUNT(*)
FROM nyc_mobility.clean.weather_silver
WHERE timezone IS NULL

UNION ALL

SELECT
    'timezone',
    'Consistency',
    COUNT(*)
FROM nyc_mobility.clean.weather_silver
WHERE TRIM(timezone) = ''

-- WEATHER CODE

UNION ALL

SELECT
    'weather_code',
    'Completeness',
    COUNT(*)
FROM nyc_mobility.clean.weather_silver
WHERE weather_code IS NULL

UNION ALL

SELECT
    'weather_code',
    'Validity',
    COUNT(*)
FROM nyc_mobility.clean.weather_silver
WHERE weather_code NOT IN (
    0,1,2,3,
    45,48,
    51,53,55,
    56,57,
    61,63,65,
    66,67,
    71,73,75,
    77,
    80,81,82,
    85,86,
    95,96,99
)

-- WIND SPEED

UNION ALL

SELECT
    'wind_speed_10m',
    'Completeness',
    COUNT(*)
FROM nyc_mobility.clean.weather_silver
WHERE wind_speed_10m IS NULL

UNION ALL

SELECT
    'wind_speed_10m',
    'Validity',
    COUNT(*)
FROM nyc_mobility.clean.weather_silver
WHERE wind_speed_10m < 0
   OR wind_speed_10m > 500

-- BUSINESS GRAIN UNIQUENESS

UNION ALL

SELECT
    'latitude, longitude, observation_time',
    'Uniqueness',
    COUNT(*)
FROM (
    SELECT
        latitude,
        longitude,
        observation_time,
        COUNT(*) AS cnt
    FROM nyc_mobility.clean.weather_silver
    GROUP BY
        latitude,
        longitude,
        observation_time
    HAVING COUNT(*) > 1
)

)

SELECT
    column_name AS `Column`,
    data_quality_check AS data_quality_check,
    failed_rows AS failed_rows,
    total_rows AS total_rows,
    ROUND(
        100.0 * failed_rows / total_rows,
        2
    ) AS `Percentage`,
    CASE
        WHEN failed_rows = 0 THEN 'PASS'
        WHEN (100.0 * failed_rows / total_rows) < 5 THEN 'WARN'
        ELSE 'FAIL'
    END AS status
FROM dq_results
CROSS JOIN total_rows
ORDER BY
    `Column`,
    data_quality_check;